# Colab: Attention + Physics-Informed Regularization for C-MAPSS (FD001)

This notebook is designed to run end-to-end with **Run all**:
- installs dependencies
- downloads NASA C-MAPSS from public mirrors
- trains baseline + physics-informed attention model
- evaluates RMSE and NASA score
- saves best checkpoint

> Note: no notebook can guarantee globally best RMSE in all environments. This setup does a reproducible best-effort search and selects the best validation checkpoint automatically.


In [ ]:
# --- 1) Environment setup (Colab) ---
!pip -q install tensorflow==2.15.1 numpy pandas scikit-learn matplotlib tqdm


In [ ]:
# --- 2) Imports and deterministic setup ---
import os
import io
import random
import zipfile
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)


In [ ]:
# --- 3) Config ---
CFG = {
    'dataset': 'FD001',
    'window_length': 30,
    'shift': 1,
    'early_rul': 125,
    'num_test_windows': 5,
    'batch_size': 256,
    'num_hiddens': 64,
    'num_layers': 2,
    'attention_size': 32,
    'dropout': 0.0,
    'learning_rate': 1e-3,
    'epochs_baseline': 25,
    'epochs_pi': 35,
    'patience': 8,
    'lambda_mono': 0.20,
    'lambda_smooth': 0.10,
    'lambda_bound': 0.05,
    'trials': 2,   # increase to 3-5 for stronger search (more time)
}

COLUMNS_TO_BE_DROPPED = [0,1,2,3,4,5,9,10,14,20,22,23]
RAW_COLUMNS = ['unit_nr','time_cycles','op_setting_1','op_setting_2','op_setting_3'] + [f's_{i}' for i in range(1,22)]


In [ ]:
# --- 4) Download NASA C-MAPSS from public source (with fallback mirrors) ---
DATA_DIR = Path('/content/cmapss_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / 'CMAPSSData.zip'

URLS = [
    # Public mirror 1
    'https://github.com/jiaxiang-cheng/PyTorch-RUL/raw/master/data/CMAPSSData.zip',
    # Public mirror 2
    'https://github.com/shining0611armor/Predicting-of-Turbofan-Engine-Degradation-Using-the-NASA-C-MAPSS-Dataset/raw/master/data/CMAPSSData.zip',
]

if not ZIP_PATH.exists():
    ok = False
    for url in URLS:
        try:
            print('Trying:', url)
            r = requests.get(url, timeout=120)
            if r.status_code == 200 and len(r.content) > 5_000_000:
                ZIP_PATH.write_bytes(r.content)
                ok = True
                print('Downloaded from:', url)
                break
        except Exception as e:
            print('Failed:', e)
    if not ok:
        raise RuntimeError('Could not download CMAPSSData.zip from available public mirrors.')
else:
    print('Using cached zip:', ZIP_PATH)

EXTRACT_DIR = DATA_DIR / 'CMAPSSData'
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)

print('Data directory contents:')
for p in sorted(DATA_DIR.rglob('*')):
    if p.is_file() and p.suffix in ['.txt', '.zip']:
        print('-', p)


In [ ]:
# --- 5) Resolve dataset file paths robustly ---
def find_file(patterns):
    for pat in patterns:
        files = list(DATA_DIR.rglob(pat))
        if files:
            return files[0]
    return None

fd = CFG['dataset']
train_path = find_file([f'train_{fd}.txt', f'train_{fd.upper()}.txt'])
test_path = find_file([f'test_{fd}.txt', f'test_{fd.upper()}.txt'])
rul_path = find_file([f'RUL_{fd}.txt', f'RUL_{fd.upper()}.txt'])

if not all([train_path, test_path, rul_path]):
    raise FileNotFoundError(f'Missing dataset files for {fd}. Found: train={train_path}, test={test_path}, rul={rul_path}')

print('train:', train_path)
print('test :', test_path)
print('rul  :', rul_path)


In [ ]:
# --- 6) Preprocessing helpers (aligned with existing repo logic) ---
def process_targets(data_length, early_rul=None):
    if early_rul is None:
        return np.arange(data_length - 1, -1, -1)
    early_rul_duration = data_length - early_rul
    if early_rul_duration <= 0:
        return np.arange(data_length - 1, -1, -1)
    return np.concatenate((np.full(shape=early_rul_duration, fill_value=early_rul), np.arange(early_rul - 1, -1, -1)))


def process_input_data_with_targets(input_data, target_data=None, window_length=1, shift=1):
    num_batches = int(np.floor((len(input_data) - window_length) / shift)) + 1
    num_features = input_data.shape[1]
    output_data = np.repeat(np.nan, repeats=num_batches * window_length * num_features).reshape(num_batches, window_length, num_features)

    if target_data is None:
        for batch in range(num_batches):
            output_data[batch, :, :] = input_data[(0 + shift * batch):(0 + shift * batch + window_length), :]
        return output_data

    output_targets = np.repeat(np.nan, repeats=num_batches)
    for batch in range(num_batches):
        output_data[batch, :, :] = input_data[(0 + shift * batch):(0 + shift * batch + window_length), :]
        output_targets[batch] = target_data[(shift * batch + (window_length - 1))]
    return output_data, output_targets


def process_test_data(test_data_for_an_engine, window_length, shift, num_test_windows=1):
    max_num_test_batches = int(np.floor((len(test_data_for_an_engine) - window_length) / shift)) + 1
    if max_num_test_batches < num_test_windows:
        required_len = (max_num_test_batches - 1) * shift + window_length
        batched = process_input_data_with_targets(test_data_for_an_engine[-required_len:, :], target_data=None, window_length=window_length, shift=shift)
        return batched, max_num_test_batches
    required_len = (num_test_windows - 1) * shift + window_length
    batched = process_input_data_with_targets(test_data_for_an_engine[-required_len:, :], target_data=None, window_length=window_length, shift=shift)
    return batched, num_test_windows


In [ ]:
# --- 7) Load and preprocess FD data ---
train_data = pd.read_csv(train_path, sep=r'\s+', header=None)
test_data = pd.read_csv(test_path, sep=r'\s+', header=None)
true_rul = pd.read_csv(rul_path, sep=r'\s+', header=None)[0].values

window_length = CFG['window_length']
shift = CFG['shift']
early_rul = CFG['early_rul']
num_test_windows = CFG['num_test_windows']

train_first_col = train_data[0]
test_first_col = test_data[0]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data.drop(columns=COLUMNS_TO_BE_DROPPED))
test_scaled = scaler.transform(test_data.drop(columns=COLUMNS_TO_BE_DROPPED))

train_df = pd.DataFrame(np.c_[train_first_col, train_scaled])
test_df = pd.DataFrame(np.c_[test_first_col, test_scaled])

num_train_machines = len(train_df[0].unique())
num_test_machines = len(test_df[0].unique())

processed_train_X, processed_train_y = [], []
train_engine_ids, train_cycle_idx = [], []

for i in range(1, num_train_machines + 1):
    temp = train_df[train_df[0] == i].drop(columns=[0]).values
    if len(temp) < window_length:
        continue
    t_targets = process_targets(len(temp), early_rul)
    x, y = process_input_data_with_targets(temp, t_targets, window_length, shift)
    processed_train_X.append(x)
    processed_train_y.append(y)
    train_engine_ids.extend([i] * len(y))
    # cycle index of each window target
    train_cycle_idx.extend(list(np.arange(window_length - 1, window_length - 1 + len(y))))

X_train_all = np.concatenate(processed_train_X).astype(np.float32)
y_train_all = np.concatenate(processed_train_y).astype(np.float32)
train_engine_ids = np.array(train_engine_ids, dtype=np.int32)
train_cycle_idx = np.array(train_cycle_idx, dtype=np.int32)

# validation split by engines (defensible split)
all_engines = np.arange(1, num_train_machines + 1)
rng = np.random.default_rng(SEED)
rng.shuffle(all_engines)
val_engines = set(all_engines[:max(10, int(0.2 * len(all_engines)))].tolist())

mask_val = np.isin(train_engine_ids, list(val_engines))
mask_tr = ~mask_val

X_tr, y_tr = X_train_all[mask_tr], y_train_all[mask_tr]
X_val, y_val = X_train_all[mask_val], y_train_all[mask_val]
eng_tr, cyc_tr = train_engine_ids[mask_tr], train_cycle_idx[mask_tr]
eng_val, cyc_val = train_engine_ids[mask_val], train_cycle_idx[mask_val]

# target scaling (as in repo approach)
target_scaler = MinMaxScaler(feature_range=(0, 1))
y_tr_s = target_scaler.fit_transform(y_tr.reshape(-1, 1)).reshape(-1).astype(np.float32)
y_val_s = target_scaler.transform(y_val.reshape(-1, 1)).reshape(-1).astype(np.float32)

# test windows
processed_test_data = []
num_test_windows_list = []
for i in range(1, num_test_machines + 1):
    temp = test_df[test_df[0] == i].drop(columns=[0]).values
    if len(temp) < window_length:
        continue
    tx, nw = process_test_data(temp, window_length, shift, num_test_windows)
    processed_test_data.append(tx)
    num_test_windows_list.append(nw)

X_test = np.concatenate(processed_test_data).astype(np.float32)

print('X_tr:', X_tr.shape, 'y_tr:', y_tr_s.shape)
print('X_val:', X_val.shape, 'y_val:', y_val_s.shape)
print('X_test:', X_test.shape, 'true_rul:', true_rul.shape)


In [ ]:
# --- 8) Build tf.data ---
BATCH = CFG['batch_size']

train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr_s, eng_tr, cyc_tr)).batch(BATCH, drop_remainder=False)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val_s, eng_val, cyc_val)).batch(BATCH, drop_remainder=False)
test_ds = tf.data.Dataset.from_tensor_slices(X_test).batch(BATCH, drop_remainder=False)


In [ ]:
# --- 9) Attention model (same family as existing repo) ---
class AdditiveAttentionForSeq(tf.keras.layers.Layer):
    def __init__(self, attention_size, **kwargs):
        super().__init__(**kwargs)
        self.attention = tf.keras.layers.Dense(attention_size)

    def call(self, state, encoder_outputs):
        seq_len = encoder_outputs.shape[1]
        flat_state = []
        for item in state:
            if isinstance(item, (list, tuple)):
                if len(item) > 0:
                    flat_state.append(item[0])
            else:
                flat_state.append(item)
        averaged_state = tf.reduce_mean(tf.stack(flat_state, axis=1), axis=1)
        state_rep = tf.repeat(tf.expand_dims(averaged_state, axis=1), repeats=seq_len, axis=1)
        concat = tf.concat((state_rep, encoder_outputs), axis=-1)
        scores = tf.nn.tanh(self.attention(concat))
        attention_weights = tf.nn.softmax(tf.reduce_sum(scores, axis=-1), axis=-1)
        return tf.matmul(tf.expand_dims(attention_weights, axis=1), encoder_outputs)


class Seq2SeqEncoder(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, num_layers, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.rnn = tf.keras.layers.RNN(
            tf.keras.layers.StackedRNNCells([
                tf.keras.layers.GRUCell(num_hiddens, dropout=dropout) for _ in range(num_layers)
            ]),
            return_sequences=True,
            return_state=True,
        )

    def call(self, x, training=False):
        output = self.rnn(x, training=training)
        return output[0], output[1:]


class Seq2SeqAttentionDecoder(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, num_layers, attention_size=32, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.rnn = tf.keras.layers.RNN(
            tf.keras.layers.StackedRNNCells([
                tf.keras.layers.GRUCell(num_hiddens, dropout=dropout) for _ in range(num_layers)
            ]),
            return_sequences=True,
            return_state=True,
        )
        self.attention = AdditiveAttentionForSeq(attention_size)
        self.dense = tf.keras.layers.Dense(1)

    def init_state(self, enc_outputs):
        outputs, hidden_state = enc_outputs
        return outputs, hidden_state

    def call(self, dec_input, state, training=False):
        enc_outputs, enc_hidden_state = state
        context = self.attention(enc_hidden_state, enc_outputs)
        rnn_input = tf.concat((dec_input, context), axis=-1)
        rnn_output = self.rnn(rnn_input, initial_state=enc_hidden_state, training=training)
        output = self.dense(tf.squeeze(rnn_output[0], axis=1))
        return output, rnn_output[1:]


class EncoderDecoder(tf.keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, enc_x, dec_x, training=False):
        enc_outputs = self.encoder(enc_x, training=training)
        dec_state = self.decoder.init_state(enc_outputs)
        return self.decoder(dec_x, dec_state, training=training)


def make_model(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    encoder = Seq2SeqEncoder(CFG['num_hiddens'], CFG['num_layers'], dropout=CFG['dropout'])
    decoder = Seq2SeqAttentionDecoder(CFG['num_hiddens'], CFG['num_layers'], CFG['attention_size'], dropout=CFG['dropout'])
    model = EncoderDecoder(encoder, decoder)
    # build
    num_features = X_tr.shape[-1]
    dummy_x = tf.zeros((1, CFG['window_length'], num_features), dtype=tf.float32)
    dummy_dec = tf.zeros((1, 1, num_features), dtype=tf.float32)
    model(dummy_x, dummy_dec, training=False)
    return model


In [ ]:
# --- 10) Physics-informed loss terms (AttnPINN-inspired regularization) ---
mse = tf.keras.losses.MeanSquaredError()


def data_loss(y_true, y_pred):
    return mse(y_true, tf.squeeze(y_pred, axis=-1))


def monotonic_loss(y_pred, eng, cyc):
    y = tf.squeeze(y_pred, axis=-1)
    idx = tf.argsort(tf.cast(eng, tf.int64) * 100000 + tf.cast(cyc, tf.int64))
    y_s = tf.gather(y, idx)
    e_s = tf.gather(eng, idx)
    de = e_s[1:] - e_s[:-1]
    dy = y_s[1:] - y_s[:-1]  # should be <= 0 over cycle
    mask = tf.cast(tf.equal(de, 0), tf.float32)
    pen = tf.nn.relu(dy) * mask
    denom = tf.reduce_sum(mask) + 1e-8
    return tf.reduce_sum(pen) / denom


def smoothness_loss(y_pred, eng, cyc):
    y = tf.squeeze(y_pred, axis=-1)
    idx = tf.argsort(tf.cast(eng, tf.int64) * 100000 + tf.cast(cyc, tf.int64))
    y_s = tf.gather(y, idx)
    e_s = tf.gather(eng, idx)

    y0, y1, y2 = y_s[:-2], y_s[1:-1], y_s[2:]
    e0, e1, e2 = e_s[:-2], e_s[1:-1], e_s[2:]
    mask = tf.cast(tf.logical_and(tf.equal(e0, e1), tf.equal(e1, e2)), tf.float32)
    sec = tf.abs(y2 - 2.0 * y1 + y0) * mask
    denom = tf.reduce_sum(mask) + 1e-8
    return tf.reduce_sum(sec) / denom


def bound_loss(y_pred):
    y = tf.squeeze(y_pred, axis=-1)
    return tf.reduce_mean(tf.nn.relu(-y) + tf.nn.relu(y - 1.0))


In [ ]:
# --- 11) Training / evaluation routines ---
def predict_scaled(model, dataset):
    preds = []
    for xb in dataset:
        enc_outputs = model.encoder(xb, training=False)
        dec_state = model.decoder.init_state(enc_outputs)
        dec_x = xb[:, -1, tf.newaxis]
        yp, _ = model.decoder(dec_x, dec_state, training=False)
        preds.append(tf.squeeze(yp, axis=-1).numpy())
    return np.concatenate(preds)


def eval_on_test(model):
    p_scaled = predict_scaled(model, test_ds)
    p = target_scaler.inverse_transform(p_scaled.reshape(-1,1)).reshape(-1)
    # clip physically plausible RUL
    p = np.clip(p, 0.0, CFG['early_rul'])

    preds_by_engine = np.split(p, np.cumsum(num_test_windows_list)[:-1])
    mean_pred = np.array([float(np.mean(v)) for v in preds_by_engine])

    rmse = np.sqrt(mean_squared_error(true_rul, mean_pred))
    mae = mean_absolute_error(true_rul, mean_pred)

    # NASA scoring function
    d = mean_pred - true_rul
    score = float(np.sum(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)))
    return {'rmse': float(rmse), 'mae': float(mae), 'score': score, 'pred': mean_pred}


def train_model(model, optimizer, train_dataset, val_dataset, epochs, use_physics, lambdas, tag='model'):
    best_rmse = np.inf
    best_weights = None
    wait = 0
    hist = []

    for ep in range(1, epochs + 1):
        tr_losses = []
        for xb, yb, eb, cb in train_dataset:
            with tf.GradientTape() as tape:
                enc_outputs = model.encoder(xb, training=True)
                dec_state = model.decoder.init_state(enc_outputs)
                dec_x = xb[:, -1, tf.newaxis]
                yp, _ = model.decoder(dec_x, dec_state, training=True)

                l_data = data_loss(yb, yp)
                if use_physics:
                    l_mono = monotonic_loss(yp, eb, cb)
                    l_smooth = smoothness_loss(yp, eb, cb)
                    l_bound = bound_loss(yp)
                    loss = l_data + lambdas['mono']*l_mono + lambdas['smooth']*l_smooth + lambdas['bound']*l_bound
                else:
                    l_mono = tf.constant(0.0)
                    l_smooth = tf.constant(0.0)
                    l_bound = tf.constant(0.0)
                    loss = l_data

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            tr_losses.append([float(loss), float(l_data), float(l_mono), float(l_smooth), float(l_bound)])

        # validation RMSE in original scale
        val_pred_scaled = predict_scaled(model, tf.data.Dataset.from_tensor_slices(X_val).batch(CFG['batch_size']))
        val_pred = target_scaler.inverse_transform(val_pred_scaled.reshape(-1,1)).reshape(-1)
        val_true = y_val
        val_rmse = float(np.sqrt(mean_squared_error(val_true, np.clip(val_pred, 0.0, CFG['early_rul']))))

        trm = np.mean(np.array(tr_losses), axis=0)
        hist.append({'epoch':ep, 'loss':trm[0], 'data':trm[1], 'mono':trm[2], 'smooth':trm[3], 'bound':trm[4], 'val_rmse':val_rmse})
        print(f"[{tag}] Ep {ep:03d} | loss={trm[0]:.5f} data={trm[1]:.5f} mono={trm[2]:.5f} smooth={trm[3]:.5f} bound={trm[4]:.5f} | val_rmse={val_rmse:.4f}")

        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_weights = model.get_weights()
            wait = 0
        else:
            wait += 1
            if wait >= CFG['patience']:
                print(f'[{tag}] Early stopping at epoch {ep}.')
                break

    if best_weights is not None:
        model.set_weights(best_weights)
    return hist, best_rmse


In [ ]:
# --- 12) Multi-trial training: baseline then physics-informed fine-tuning ---
results = []
CHECKPOINT_DIR = Path('/content/attnpinn_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for trial in range(CFG['trials']):
    seed = SEED + trial * 13
    print('
' + '='*80)
    print(f'TRIAL {trial+1}/{CFG["trials"]} | seed={seed}')
    print('='*80)

    model = make_model(seed=seed)
    opt = tf.keras.optimizers.Adam(learning_rate=CFG['learning_rate'])

    # Stage A: data-only baseline warmup
    hist_base, best_val_base = train_model(
        model, opt, train_ds, val_ds,
        epochs=CFG['epochs_baseline'],
        use_physics=False,
        lambdas={'mono':0.0,'smooth':0.0,'bound':0.0},
        tag=f'trial{trial+1}-baseline'
    )

    # Stage B: physics-informed fine-tuning
    hist_pi, best_val_pi = train_model(
        model, opt, train_ds, val_ds,
        epochs=CFG['epochs_pi'],
        use_physics=True,
        lambdas={'mono':CFG['lambda_mono'], 'smooth':CFG['lambda_smooth'], 'bound':CFG['lambda_bound']},
        tag=f'trial{trial+1}-pi'
    )

    test_metrics = eval_on_test(model)
    ckpt_stem = CHECKPOINT_DIR / f"FD001_attnpinn_trial{trial+1}_val{best_val_pi:.4f}_rmse{test_metrics['rmse']:.4f}"
    tf.train.Checkpoint(model=model).save(str(ckpt_stem))

    results.append({
        'trial': trial+1,
        'seed': seed,
        'best_val_base': best_val_base,
        'best_val_pi': best_val_pi,
        'test_rmse': test_metrics['rmse'],
        'test_mae': test_metrics['mae'],
        'test_score': test_metrics['score'],
        'ckpt': str(ckpt_stem),
        'hist_base': hist_base,
        'hist_pi': hist_pi,
    })


In [ ]:
# --- 13) Select best trial and summarize ---
res_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ['hist_base','hist_pi']} for r in results]).sort_values('test_rmse')
res_df


In [ ]:
# --- 14) Report best model metrics + save concise artifacts ---
best = sorted(results, key=lambda x: x['test_rmse'])[0]
print('Best trial:', best['trial'])
print('Seed      :', best['seed'])
print('Val RMSE  :', best['best_val_pi'])
print('Test RMSE :', best['test_rmse'])
print('Test MAE  :', best['test_mae'])
print('NASA Score:', best['test_score'])
print('Checkpoint:', best['ckpt'])

summary_path = CHECKPOINT_DIR / 'best_result_summary.csv'
pd.DataFrame([{
    'trial': best['trial'],
    'seed': best['seed'],
    'best_val_rmse': best['best_val_pi'],
    'test_rmse': best['test_rmse'],
    'test_mae': best['test_mae'],
    'test_score': best['test_score'],
    'checkpoint': best['ckpt'],
}]).to_csv(summary_path, index=False)

print('Saved summary to', summary_path)


In [ ]:
# --- 15) Optional: export final model weights in repo-compatible checkpoint format ---
# If you want to reuse in your app/repo, copy these files from Colab to Drive/Git repo.
BEST_EXPORT_DIR = Path('/content/attnpinn_export')
BEST_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# rebuild model and restore best trial checkpoint
best_ckpt = tf.train.latest_checkpoint(str(Path(best['ckpt']).parent)) if False else best['ckpt']
# best['ckpt'] is already the save stem; restore directly
model_export = make_model(seed=best['seed'])
status = tf.train.Checkpoint(model=model_export).restore(best['ckpt'])
status.expect_partial()

export_stem = BEST_EXPORT_DIR / 'FD001_attnpinn_best'
tf.train.Checkpoint(model=model_export).save(str(export_stem))
print('Exported checkpoint stem:', export_stem)


## Next step back in your repo

- Copy the best checkpoint files from Colab output into your repository (e.g., `saved_models/cmapss/attnpinn_fd001/`).
- Update your local inference loader to optionally use this new checkpoint.
- Keep baseline and PI checkpoints both available for side-by-side demo.
